# LegalLens — Milestone 2: Exploratory Data Analysis (EDA)
### NLP-Based Classifier for Indian Court Judgments (Civil vs. Criminal)

This notebook provides exploratory analysis of the **Jud-IPL** dataset for whole-judgment classification:
- Empirical dataset scale & label distribution
- Document length distributions (words, characters, percentiles)
- Class-wise length comparison (Civil vs. Criminal)
- Duplicate cluster forensics & conflicting-label case examination
- Discriminative vocabulary & n-gram frequency extraction
- Text-quality artifacts & export truncation limits

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Paths
DATA_PATH = Path('../data/raw/case_files_total.csv')
METRICS_PATH = Path('../outputs/metrics/eda_metrics.json')

with open(METRICS_PATH, 'r') as f:
    eda_metrics = json.load(f)

print('Loaded EDA Metrics Summary successfully!')

## 1. Dataset Scale & Class Distribution
Summary of raw counts, excluded rows (missing labels / empty text), and usable dataset.

In [ ]:
counts = eda_metrics['summary_counts']
pd.DataFrame({
    'Metric': [
        'Total Raw Rows', 'Labeled Rows (Civil/Criminal)', 
        'Excluded: Missing Category', 'Excluded: Empty Judgement Text',
        'Usable Non-Empty Judgments', 'Civil Cases', 'Criminal Cases', 'Imbalance Ratio (Civil:Criminal)'
    ],
    'Count / Value': [
        f"{counts['raw_total']:,}", f"{counts['labeled_total']:,}",
        f"{counts['excluded_unlabeled']:,}", f"{counts['excluded_empty_text']:,}",
        f"{counts['valid_nonempty_total']:,}", f"{counts['civil_count']:,} ({counts['civil_percentage']}%)",
        f"{counts['criminal_count']:,} ({counts['criminal_percentage']}%)", f"{counts['imbalance_ratio']}:1"
    ]
})

## 2. Document Length Comparison: Civil vs. Criminal
Indian court judgments are characteristically lengthy. Here we compare word and character lengths across legal branches.

In [ ]:
lengths = eda_metrics['document_lengths']
len_df = pd.DataFrame({
    'Metric': ['Min Words', '25th Pct', 'Median Words', 'Mean Words', '75th Pct', '90th Pct', '95th Pct', 'Max Words'],
    'Overall': [lengths['overall_words']['min'], lengths['overall_words']['p25'], lengths['overall_words']['median'], lengths['overall_words']['mean'], lengths['overall_words']['p75'], lengths['overall_words']['p90'], lengths['overall_words']['p95'], lengths['overall_words']['max']],
    'Civil': [lengths['civil_words']['min'], lengths['civil_words']['p25'], lengths['civil_words']['median'], lengths['civil_words']['mean'], lengths['civil_words']['p75'], lengths['civil_words']['p90'], lengths['civil_words']['p95'], lengths['civil_words']['max']],
    'Criminal': [lengths['criminal_words']['min'], lengths['criminal_words']['p25'], lengths['criminal_words']['median'], lengths['criminal_words']['mean'], lengths['criminal_words']['p75'], lengths['criminal_words']['p90'], lengths['criminal_words']['p95'], lengths['criminal_words']['max']],
})
len_df

## 3. Duplicate Cluster Analysis & Conflicting Label Forensics

In [ ]:
dups = eda_metrics['duplicate_analysis']
print(f"Unique text hashes: {dups['unique_text_hashes']:,}")
print(f"Distinct duplicate clusters: {dups['distinct_duplicate_groups']:,}")
print(f"Excess duplicate rows: {dups['excess_duplicate_records']:,}")
print(f"Same-label clusters: {dups['same_label_groups']:,}")
print(f"Conflicting-label clusters: {dups['conflicting_label_groups']}")

# Conflicting cases inspection
for c in dups['conflicting_case_forensics']:
    print('=' * 60)
    print(f"{c['conflict_id']}: Length {c['char_length']} chars, {c['word_length']} words")
    print("Records in cluster:", c['records'])
    print("Preview:", c['text_preview'][:200])

## 4. Substantive Discriminative Legal N-Grams
Top domain bigrams extracted for Civil and Criminal categories.

In [ ]:
vocab = eda_metrics['vocabulary_and_ngrams']
pd.DataFrame({
    'Top Civil Bigrams': vocab['top_bigrams_civil'][:10],
    'Top Criminal Bigrams': vocab['top_bigrams_criminal'][:10]
})